# Portfolio Management

A portfolio is a collection of financial investments. Let's see how we construct, analyse and optimise portfolios with Python. Like with every notebook, this one starts with a list of imports.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

plt.style.use("ggplot")

## Portfolio Formation

A portfolio is an investment made up of multiple securities, where each individual security can have a different *weight*. 

We'll begin with what is known as a 1/N portfolio, with 5 stocks in it. The 1/N refers to the total investment being divided equally among the N stocks that comprise the portfolio - which will be 5 in our case today. We also refer to this as an equal-weighted portfolio.

Depending on what kind of data we have, portfolio formation can involve different steps. 

In one case, you might have pulled some data using the [College WRDS access](https://www.imperial.ac.uk/admin-services/library/subject-support/business/crsp/), or obtained a CSV with panel data in some other way, such as what we are given in `SP500_2025.csv`.

### Exercise: Filter a portfolio

Load the data from `SP500_2025.csv` and create a new dataframe with only the following stocks:

- TSLA
- MSFT
- AMZN
- META
- NVDA

**HINT** You will need to call the `.isin()` method on your ticker column, passing it a list of your desired tickers.

In [ ]:
## YOUR CODE GOES HERE

Alternatively, you can pull financial data for a portfolio directly from Yahoo! Finance, using `yfinance`.

In [ ]:
# Decide on your tickers
tickers = ["TSLA", "MSFT", "AMZN", "META", "NVDA"]

# Download them across some date range
data = yf.download(tickers, start="2025-01-01", end="2025-12-31")

# Check the columns - they are "multiindex"
display(data.columns)

# Check the names of the elements in the multiindex
data.columns.names

# Stack - or pull an element from the multiindex
df = data.stack(level="Ticker").reset_index().sort_values(["Ticker", "Date"], ignore_index=True)

# Save our work
df.to_csv("my_portfolio.csv")

Pivoted dataframes have an ideal shape for portfolio work. Let's pivot our downloaded portfolio and assign our weights for the 1/N portfolio.

In [ ]:
# Pivot the DataFrame to wide format with stocks as columns and dates as index
portfolio = df.pivot(index='Date', columns='Ticker', values='Close')
display(portfolio)

weights = np.ones(5) / 5
weights

## Calculating Portfolio Daily Returns

First, let's determine the returns of our portfolio. We'll need to start with the daily returns of each stock, and then use `.dot()` to mutliply each column of returns against its weight in the portfolio.

**Notice we're using `.dot()` from Pandas rather than `np.dot()` directly. This is the safer choice for our portfolio work**

In [ ]:
# Calculate the simple return of each stock in the portfolio
returns = portfolio.pct_change()
returns = returns.dropna()

# Compute the portfolio returns
portfolio_returns = returns.dot(weights)

# While we're at it check the daily volatility (standard deviation of portfolio returns)
daily_volatility = portfolio_returns.std()

## Annualising Returns and Volatility

To provide a broader picture of our portfolio's performance, let's calculate the annualised portfolio return and volatility. These are important metrics for evaluating and discussing portfolios. They are also used frequently in more advanced financial analyses.

In [ ]:
# Define the number of trading days in a year
trading_days = 252

# Annualise the portfolio return
annualised_return = portfolio_returns.mean() * trading_days

# Annualise the portfolio volatility
annualised_volatility = daily_volatility * np.sqrt(trading_days)

## Sharpe Ratio

The Sharpe Ratio is a measure that helps investors understand the risk-adjusted return of an investment. A high Sharpe ratio indicates that the portfolio's returns are higher for each unit of risk taken on. In contrast, a lower Sharpe Ratio indicates a less favorable risk-reward trade-off, with the potential for lower returns relative to the amount of risk being assumed.

The Sharpe Ratio is calculated using annualised portfolio returns, portfolio volatility (as the measure of risk), and the *risk-free rate*. The risk-free rate is often derived from the yield of a theoretically risk-free investment, typically a government bond. The Sharpe Ratio is the average return earned in excess of the risk-free rate per unit of volatility or total risk.

### Exercise: Looking Sharpe

Calculate the Sharpe Ratio for our portfolio by applying the formula below.

$$ \text{Sharpe Ratio} = \frac{R_p - R_f}{\sigma_p} $$

- $R_p$ is our annualised portfolio return
- $R_f$ is the risk-free rate
- $\sigma_p$ is our annualised volatility


In [ ]:
## YOUR CODE GOES HERE